## <U>Etapa 1</U>: Importar bibliotecas y cargar datos

### Importar bibliotecas

In [1]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import time
import tracemalloc

print("Bibliotecas cargadas correctamente")

Bibliotecas cargadas correctamente


### Cargar los datos de entrenamiento y prueba

In [3]:
X_train = pd.read_csv(
    "../Datos/Procesados/perfiles_entrenamiento.csv"
)

X_test = pd.read_csv(
    "../Datos/Procesados/perfiles_prueba.csv"
)

print("Dimensiones de entrenamiento:", X_train.shape)
print("Dimensiones de prueba:", X_test.shape)

display(X_train.head().round(2))

Dimensiones de entrenamiento: (800, 12)
Dimensiones de prueba: (200, 12)


,interes_salud,interes_tecnologia,interes_educacion,interes_arte,interes_social,interes_ambiente,fortaleza_matematica,fortaleza_lengua,fortaleza_biologia,fortaleza_historia,fortaleza_fisica,fortaleza_tecnologia
0,0.5,0.75,0.75,1.00,0.25,0.50,0.14,0.56,0.36,0.51,0.36,0.29
1,0.5,0.25,0.50,0.50,0.50,0.25,0.09,0.56,0.43,0.13,0.30,0.23
2,0.5,0.50,1.00,1.00,0.50,1.00,0.79,0.62,0.69,0.65,1.00,0.95
3,1.0,0.50,1.00,0.75,1.00,0.75,1.00,0.91,0.89,0.93,0.81,0.74
4,0.5,0.75,0.50,0.75,0.75,1.00,0.44,0.43,0.58,0.63,0.30,0.40


## <U>Etapa 2</U>: Evaluar distintos valores de K

In [4]:
resultados_k = []

for k in range(2, 9):

    modelo = KMeans(n_clusters=k,random_state=42,n_init=10)

    etiquetas = modelo.fit_predict(X_train)

    inercia = modelo.inertia_
    silueta = silhouette_score(X_train, etiquetas)

    resultados_k.append({
        "Cantidad de grupos (K)": k,
        "Inercia": inercia,
        "Silhouette Score": silueta
    })

df_resultados_k = pd.DataFrame(resultados_k)

display(df_resultados_k.round(2))

,Cantidad de grupos (K),Inercia,Silhouette Score
0,2,550.91,0.18
1,3,508.14,0.13
2,4,478.44,0.13
3,5,451.30,0.12
4,6,424.80,0.13
5,7,406.52,0.13
6,8,391.25,0.13


## <U>Etapa 3</U>: Medir tiempo y memoria del entrenamiento

In [5]:
tracemalloc.start()

inicio = time.perf_counter()

modelo_final = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

modelo_final.fit(X_train)

tiempo_entrenamiento = time.perf_counter() - inicio

memoria_actual, memoria_maxima = tracemalloc.get_traced_memory()

tracemalloc.stop()


### Predecir los grupos de los perfiles de prueba

In [6]:
inicio = time.perf_counter()

etiquetas_test = modelo_final.predict(X_test)

tiempo_inferencia = time.perf_counter() - inicio


### Evaluar la separacion de los grupos

In [7]:
silhouette_test = silhouette_score(
    X_test,
    etiquetas_test
)

### Comprobacion

In [8]:
print("Cantidad de grupos:", modelo_final.n_clusters)
print("Inercia de entrenamiento:", round(modelo_final.inertia_, 2))
print("Silhouette en prueba:", round(silhouette_test, 4))

print("\nTiempo de entrenamiento:", round(tiempo_entrenamiento, 4), "segundos")
print("Tiempo de inferencia:", round(tiempo_inferencia, 6), "segundos")
print("Memoria máxima registrada:", round(memoria_maxima / 1024**2, 2), "MB")

Cantidad de grupos: 2
Inercia de entrenamiento: 550.91
Silhouette en prueba: 0.1722

Tiempo de entrenamiento: 0.1852 segundos
Tiempo de inferencia: 0.013702 segundos
Memoria máxima registrada: 0.28 MB


## <U>Etapa 4</U>: Conocer los grupos

### Crear una tabla con grupos asignados

In [9]:
df_grupos = X_test.copy()

df_grupos["grupo"] = etiquetas_test

print("Tabla creada")

Tabla creada


### Cantidad de perfiles por grupo

In [10]:
print("Cantidad de perfiles por grupo:")
display(df_grupos["grupo"].value_counts().sort_index())

Cantidad de perfiles por grupo:


grupo
0     91
1    109
Name: count, dtype: int64

### Promedio de cada variables por grupo

In [11]:
print("\nCaracterísticas promedio de cada grupo:")
display(
    df_grupos.groupby("grupo")[X_test.columns]
    .mean()
    .round(2)
)


Características promedio de cada grupo:


,interes_salud,interes_tecnologia,interes_educacion,interes_arte,interes_social,interes_ambiente,fortaleza_matematica,fortaleza_lengua,fortaleza_biologia,fortaleza_historia,fortaleza_fisica,fortaleza_tecnologia
grupo,,,,,,,,,,,,
0,0.29,0.23,0.21,0.25,0.26,0.31,0.50,0.58,0.57,0.56,0.60,0.54
1,0.62,0.60,0.60,0.61,0.54,0.54,0.55,0.63,0.62,0.58,0.63,0.55


## <U>Etapa 5</U>: Conclusiones del modelo K-Means
### Se aplicó el algoritmo K-Means utilizando 800 perfiles sintéticos para el entrenamiento y 200 para la prueba, considerando 12 variables relacionadas con intereses personales y fortalezas académicas.

### Después de evaluar distintas cantidades de grupos (K), se seleccionó K=2 debido a que obtuvo el mayor Silhouette Score (0,18).

### En el conjunto de prueba se identificaron dos grupos:

### Grupo 0: 91 perfiles.

### Grupo 1: 109 perfiles.

### El análisis de los promedios mostró que el Grupo 1 presenta niveles generalmente superiores de intereses y fortalezas académicas respecto al Grupo 0.

### Sin embargo, el Silhouette Score de 0,1722 indica una separación débil entre los grupos. Esto sugiere que el modelo identifica diferencias generales en los niveles de las variables, pero no permite establecer perfiles vocacionales claramente diferenciados.

### Por lo tanto, K-Means se utilizará como herramienta exploratoria complementaria, mientras que el recomendador basado en afinidades entre intereses, fortalezas y áreas de conocimiento será el componente principal del sistema.

## **Limitación: los resultados se obtuvieron con datos sintéticos, por lo que no representan conclusiones validadas sobre estudiantes reales.**